# 03 · Évaluation : scores, vérification, analyse des erreurs

Dernière étape : on mesure ce que la reconstruction a retrouvé.

1. **Scores** : chaque reconstruction est comparée à la note (*note vs recon*) et au gold (*source vs recon*), en exact et en sémantique
2. **Vérification** : MedGemma relit chaque reconstruction et juge si elle est fidèle à la note
3. **Analyse des erreurs** : pour chaque patient, à quelle étape l'information s'est perdue
4. **Synthèses et exports** : tables CSV, et en option publication dans `results/` avec les figures

Il faut avoir lancé le notebook 02. Les scores sémantiques et la vérification appellent MedGemma ; chaque
réponse est mise en cache, donc relancer le notebook ne coûte presque rien.

In [ ]:
# Sur Google Colab seulement : récupérer le dépôt et installer les dépendances
# !git clone https://github.com/brettsamy-database/medical-data-reconstruction.git bupa
# %cd bupa/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Pour pouvoir importer `config` et `src`, que le notebook soit lancé depuis notebooks/ ou depuis la racine
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display

from config import settings

settings.ensure_dirs()
pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

In [ ]:
# MedGemma est un modèle « gated » : il faut un token Hugging Face.
# En local, le mettre dans un fichier .env à la racine (HF_TOKEN=hf_..., voir .env.example) ;
# sur Colab, dans un secret nommé HF_TOKEN.
from src.llm import get_llm, hf_login

hf_login()
llm = get_llm()  # le modèle n'est chargé qu'au premier appel
print("Model:", settings.MODEL_NAME, "| transformers/torch available:", llm.available())

## Charger les sorties du notebook 02

In [ ]:
from src import gold, notes, reconstruction

patients = gold.load_gold_patients()
raw_notes, generated_notes = notes.load_notes()
recon_raw, recon_clean, recon_clipped = reconstruction.load_reconstructions()
print(f"{len(patients)} patients | {len(generated_notes)} notes | {len(recon_clean)} reconstructions")

### Repartir de zéro (optionnel)

Supprime les scores, la vérification et les exports, mais garde le gold, les notes et les reconstructions.
Utile si on a modifié une métrique ou un prompt d'évaluation.

In [ ]:
from src import export

RESET_BENCHMARK_OUTPUTS = False
if RESET_BENCHMARK_OUTPUTS:
    removed = export.reset_benchmark_outputs()
    print("Removed files:", len(removed))
else:
    print("RESET_BENCHMARK_OUTPUTS = False -> no file removed.")

## 1. Scores

- **note vs recon** : le modèle a-t-il bien extrait ce qui est écrit dans la note ?
- **source vs recon** : que reste-t-il du dossier d'origine ? Cette comparaison cumule les pertes de la génération et de l'extraction.

Chaque comparaison est faite en **exact** (même libellé après normalisation) et en **sémantique** (le LLM juge si
deux libellés désignent la même chose, par exemple « anemia » et « Anemia (disorder) »).

In [ ]:
from src import evaluation

df_scores = evaluation.score_all(patients, recon_clean, recon_raw, generated_notes, llm=llm)
display(df_scores)
evaluation.print_mean_scores(df_scores)

## 2. Vérification

In [ ]:
from src import verification

df_verification, df_verification_by_note_style = verification.run_verification(patients, recon_clean, generated_notes, llm=llm)
df_scores = verification.merge_verification_into_scores(df_scores, df_verification)
display(df_verification)
display(df_verification_by_note_style)

## 3. Analyse des erreurs

Les scores disent à quel point la reconstruction est bonne ; l'analyse des erreurs dit où elle a échoué.
Pour chaque patient, et séparément pour les conditions et les prescriptions :

- `overall` : le type d'erreur dominant (`perfect_reconstruction`, `exact_wording_gap_only`, `source_to_note_gap_or_generation_loss`, `under_extraction_from_note`…)
- `stage` : l'étape où elle se produit (`source_to_note`, `note_to_reconstruction`, `canonicalization_exact_matching`, `mixed`, `none`)
- `severity` : `none`, `mild`, `moderate` ou `severe`
- `explanation` : les chiffres qui ont mené à ce classement

Les règles sont détaillées dans `classify_error_type` (`src/evaluation.py`).

In [ ]:
tables = evaluation.run_error_analysis(df_scores)
df_scores = tables["df_scores"]
display(tables["df_error_analysis"])
display(tables["df_error_types_by_note_style_conditions"])
display(tables["df_error_types_by_note_style_medication_requests"])
print("Hardest patients by note_style")
display(tables["df_hard_cases_by_note_style"])

## 4. Tables de synthèse

In [ ]:
df_scores_overall_summary, df_scores_summary_by_note_style = evaluation.run_summaries(df_scores, df_verification)
display(df_scores_overall_summary)
display(df_scores_summary_by_note_style)

## 5. Exports

Toutes les tables sont reconstruites depuis le disque et écrites dans `data/processed/exports/`, avec une archive zip complète.

In [ ]:
exports = export.rebuild_datasets_from_cache()
df_export_review = export.export_datasets(exports)
display(df_export_review)

## 6. Publier les résultats

Cette cellule remplace les CSV de `results/` et les figures de `assets/figures/`. À n'activer que pour un run à publier.

In [ ]:
from src import plots

PUBLISH = False
if PUBLISH:
    written = export.publish_results()
    print(len(written), "CSV written to results/")
    for p in plots.make_all_figures():
        print("Figure:", p)

In [ ]:
# Regarder un patient en détail
pid = exports["df_gold"].iloc[0]["patient_id"]
print(exports["df_notes"].set_index("patient_id").loc[pid, "note_text"])
exports["df_recon"].set_index("patient_id").loc[pid, ["raw_conditions", "conditions", "conditions_clipped"]].to_dict()